In [6]:
import numpy as np
import pandas as pd


# ============================================
# DATASET: Play Tennis
# ============================================
# Création du dictionnaire de données
data = {
    'Outlook': ['Sunny', 'Sunny', 'Overcast', 'Rain', 'Rain', 'Rain', 'Overcast', 
                'Sunny', 'Sunny', 'Rain', 'Sunny', 'Overcast', 'Overcast', 'Rain'],
    'Temp': ['Hot', 'Hot', 'Hot', 'Mild', 'Cool', 'Cool', 'Cool', 
             'Mild', 'Cool', 'Mild', 'Mild', 'Mild', 'Hot', 'Mild'],
    'Humidity': ['High', 'High', 'High', 'High', 'Normal', 'Normal', 'Normal', 
                 'High', 'Normal', 'Normal', 'Normal', 'High', 'Normal', 'High'],
    'Wind': ['Weak', 'Strong', 'Weak', 'Weak', 'Weak', 'Strong', 'Strong', 
             'Weak', 'Weak', 'Weak', 'Strong', 'Strong', 'Weak', 'Strong'],
    'Play': ['No', 'No', 'Yes', 'Yes', 'Yes', 'No', 'Yes', 
             'No', 'Yes', 'Yes', 'Yes', 'Yes', 'Yes', 'No']
}

# Transformation en DataFrame (Tableau)
df = pd.DataFrame(data)

# Séparation des caractéristiques (X) et de la cible (y)
X = df.drop('Play', axis=1).values
y = df['Play'].values
feature_names = list(df.drop('Play', axis=1).columns)

print("Dataset chargé avec succès !")
print(f"Nombre de lignes : {len(df)}")


# ============================================
# ALGORITHME ID3
# ============================================
### 1- calculation de l'entropie 
def entropy(self, y):
    # On calcule la probabilité de chaque classe (ex: P(Oui) et P(Non))
    # np.mean(y == c) donne la proportion d'éléments de la classe 'c'
    probs = [np.mean(y == c) for c in np.unique(y)]
    
    # On applique la formule mathématique : H(S) = - Somme(p * log2(p))
    # Le "if p > 0" évite l'erreur mathématique du log(0)
    return -np.sum([p * np.log2(p) for p in probs if p > 0])
    
### 2- calculation de le gain
def information_gain(self, X, y, feature_idx):
    # 1. On calcule l'entropie totale du groupe actuel (avant la division)
    total_entropy = self.entropy(y)
    
    # 2. On identifie les valeurs possibles de la caractéristique (ex: Soleil, Pluie, Nuageux)
    values, counts = np.unique(X[:, feature_idx], return_counts=True)
    
    # 3. On calcule l'entropie pondérée des sous-groupes créés après la division
    weighted_entropy = np.sum([
        (counts[i] / np.sum(counts)) * self.entropy(y[X[:, feature_idx] == values[i]])
        for i in range(len(values))
    ])
    
    # 4. Gain = Entropie Initiale - Entropie après division
    return total_entropy - weighted_entropy

### 3- fonction récursive : transforme les données brutes en un arbre de décision logique    
def _build_tree(self, X, y, features):
    # CAS D'ARRÊT 1 : Si toutes les étiquettes sont les mêmes (ex: que des "Oui")
    # On crée une feuille avec cette réponse.
    if len(np.unique(y)) <= 1:
        return np.unique(y)[0]

    # CAS D'ARRÊT 2 : Si on n'a plus de caractéristiques pour diviser
    # On retourne la classe majoritaire (le vote du plus grand nombre).
    if len(features) == 0:
        return np.unique(y)[np.argmax(np.unique(y, return_counts=True)[1])]

    # ÉTAPE DE DIVISION :
    # On calcule le Gain d'Information pour chaque caractéristique restante
    gains = [self.information_gain(X, y, i) for i in range(len(features))]
    # On choisit l'index de celle qui a le plus haut gain
    best_feature_idx = np.argmax(gains)
    best_feature_name = features[best_feature_idx]

    # On crée un nœud (dictionnaire) pour cette caractéristique
    tree = {best_feature_name: {}}
    
    # Pour chaque valeur possible de cette caractéristique (ex: "Soleil")
    feature_values = np.unique(X[:, best_feature_idx])
    for val in feature_values:
        # On filtre les données qui possèdent cette valeur
        sub_X = X[X[:, best_feature_idx] == val]
        sub_y = y[X[:, best_feature_idx] == val]
        
        # On retire la caractéristique utilisée de la liste pour la suite
        new_features = [f for i, f in enumerate(features) if i != best_feature_idx]
        
        # RÉCURSION : On recommence le processus pour créer la branche suivante
        tree[best_feature_name][val] = self._build_tree(
            np.delete(sub_X, best_feature_idx, axis=1), sub_y, new_features
        )
            
    return tree


### 4- phse de test 
def predict(self, sample, tree=None):
    # Si le nœud actuel n'est pas un dictionnaire, c'est une feuille (la réponse finale)
    if not isinstance(tree, dict):
        return tree
    
    # On regarde quelle caractéristique est testée à ce nœud
    feature = list(tree.keys())[0]
    # On regarde la valeur de cette caractéristique dans l'exemple à tester
    val = sample[feature]
    
    # On suit la branche correspondante
    if val in tree[feature]:
        return self.predict(sample, tree[feature][val])
    else:
        return None # Cas où la valeur n'a jamais été vue pendant l'entraînement

Dataset chargé avec succès !
Nombre de lignes : 14
